In [15]:
pip install finlight-client


In [20]:
from finlight_client import FinlightApi, ApiConfig
from finlight_client.models import GetArticlesParams

API_KEY = "sk_5f66a42825729595ee752f8b92efe43dc948b97c23aa8e6f60f716450ac824ba"
config = ApiConfig(api_key=API_KEY)
client = FinlightApi(config)



Fetching AI-enriched market data for Nvidia...

--- Finlight Automated Intelligence Dashboard ---

[1] Title: Artificial Intelligence (AI) Is Moving Beyond Data Centers. Nvidia Has Already Turned This Opportunity Into a Multibillion-Dollar Business
    Date Published: 2026-05-29 13:43:00+00:00
    AI Sentiment Tag: None

[2] Title: Nvidia Stock Is Stuck. Taiwan Tech Conference Could Be a Catalyst.
    Date Published: 2026-05-29 12:49:00+00:00
    AI Sentiment Tag: None

[3] Title: Dell Raises Outlook as AI Server Demand Surges
    Date Published: 2026-05-29 12:22:30+00:00
    AI Sentiment Tag: None

[4] Title: Is Nvidia a Top Dividend Stock?
    Date Published: 2026-05-29 10:50:00+00:00
    AI Sentiment Tag: None

[5] Title: Jim Cramer on NVIDIA: “The King Has Indeed Begun to Underperform When It Reports”
    Date Published: 2026-05-29 10:49:55+00:00
    AI Sentiment Tag: None

[6] Title: Dell rallies about 40% on strong Nvidia‑powered AI server demand
    Date Published: 2026-05-29 09

In [22]:

print("Fetching AI-enriched market data for Nvidia...")

# 2. Build finance-specific query parameters
params = GetArticlesParams(
    query="Nvidia",
    language="en",
    includeContent=True
)

# 3. Execute the API fetch
try:
    response = client.articles.fetch_articles(params=params)
except Exception as e:
    print(f"Connection error: {e}")
    response = None

# 4. Display Finlight's built-in NLP data (UPDATED COMPONENT)
if response and response.articles:
    print("\n--- Finlight Automated Intelligence Dashboard ---")

    for idx, article in enumerate(response.articles, 1):
        print(f"\n[{idx}] Title: {article.title}")
        print(f"    Date Published: {article.publishDate}")

        # Display natively tagged stock tickers
        if hasattr(article, 'companies') and article.companies:
            tickers = [c.ticker for c in article.companies]
            print(f"    Detected Tickers: {', '.join(tickers)}")

        # EXTRACT NESTED SENTIMENT (Fixes the 'None' issue)
        if getattr(article, 'sentiment', None) is not None:
            sentiment_obj = article.sentiment

            # Extract internal label and confidence score
            label = getattr(sentiment_obj, 'label', 'UNKNOWN')
            score = getattr(sentiment_obj, 'score', 0.0)

            # Map labels to student-friendly indicators
            visual_map = {"POSITIVE": "🟢 BULLISH", "NEGATIVE": "🔴 BEARISH", "NEUTRAL": "🟡 NEUTRAL"}
            emoji_label = visual_map.get(label.upper(), "⚪ UNKNOWN")

            print(f"    AI Sentiment Tag: {emoji_label} (Confidence: {score:.2f})")
        else:
            print("    AI Sentiment Tag: ⚪ No Sentiment Generated")
else:
    print("No articles returned or invalid API key configuration.")


Fetching AI-enriched market data for Nvidia...

--- Finlight Automated Intelligence Dashboard ---

[1] Title: Artificial Intelligence (AI) Is Moving Beyond Data Centers. Nvidia Has Already Turned This Opportunity Into a Multibillion-Dollar Business
    Date Published: 2026-05-29 13:43:00+00:00
    AI Sentiment Tag: ⚪ No Sentiment Generated

[2] Title: Nvidia Stock Is Stuck. Taiwan Tech Conference Could Be a Catalyst.
    Date Published: 2026-05-29 12:49:00+00:00
    AI Sentiment Tag: ⚪ No Sentiment Generated

[3] Title: Dell Raises Outlook as AI Server Demand Surges
    Date Published: 2026-05-29 12:22:30+00:00
    AI Sentiment Tag: ⚪ No Sentiment Generated

[4] Title: Is Nvidia a Top Dividend Stock?
    Date Published: 2026-05-29 10:50:00+00:00
    AI Sentiment Tag: ⚪ No Sentiment Generated

[5] Title: Jim Cramer on NVIDIA: “The King Has Indeed Begun to Underperform When It Reports”
    Date Published: 2026-05-29 10:49:55+00:00
    AI Sentiment Tag: ⚪ No Sentiment Generated

[6] Title

The ⚪ No Sentiment Generated fallback is occurring because the Finlight backend hasn't populated the sentiment field for these specific news articles yet. When you use Finlight's raw /articles search endpoint, the system aggregates global feeds instantly, but its internal NLP machine learning models can sometimes take a few minutes to process, calculate, and append the sentiment blocks to the newest articles.

To fix this we will modify Section 4 of the code to calculate an immediate fallback sentiment score using textblob whenever Finlight's native AI tag is missing.

In [23]:
from textblob import TextBlob

# 4. Display Finlight's built-in NLP data with local NLP fallback
if response and response.articles:
    print("\n--- Finlight Automated Intelligence Dashboard ---")

    for idx, article in enumerate(response.articles, 1):
        print(f"\n[{idx}] Title: {article.title}")
        print(f"    Date Published: {article.publishDate}")

        # Display natively tagged stock tickers
        if hasattr(article, 'companies') and article.companies:
            tickers = [c.ticker for c in article.companies]
            print(f"    Detected Tickers: {', '.join(tickers)}")

        # EXTRACT NATIVE SENTIMENT OR APPLY LOCAL NLP FALLBACK
        sentiment_obj = getattr(article, 'sentiment', None)

        if sentiment_obj is not None:
            # Case A: Use Finlight's native server-side AI calculation
            label = getattr(sentiment_obj, 'label', 'UNKNOWN')
            score = getattr(sentiment_obj, 'score', 0.0)

            visual_map = {"POSITIVE": "🟢 BULLISH", "NEGATIVE": "🔴 BEARISH", "NEUTRAL": "🟡 NEUTRAL"}
            emoji_label = visual_map.get(label.upper(), "⚪ UNKNOWN")
            print(f"    AI Sentiment Tag: {emoji_label} (Finlight Native AI | Confidence: {score:.2f})")

        else:
            # Case B: Fallback to local rule-based NLP if server-side data isn't ready
            analysis = TextBlob(article.title)
            local_polarity = analysis.sentiment.polarity

            if local_polarity > 0.05:
                emoji_label = "🟢 BULLISH"
            elif local_polarity < -0.05:
                emoji_label = "🔴 BEARISH"
            else:
                emoji_label = "🟡 NEUTRAL"

            print(f"    AI Sentiment Tag: {emoji_label} (Local TextBlob Fallback | Polarity: {local_polarity:.2f})")
else:
    print("No articles returned or invalid API key configuration.")



--- Finlight Automated Intelligence Dashboard ---

[1] Title: Artificial Intelligence (AI) Is Moving Beyond Data Centers. Nvidia Has Already Turned This Opportunity Into a Multibillion-Dollar Business
    Date Published: 2026-05-29 13:43:00+00:00
    AI Sentiment Tag: 🔴 BEARISH (Local TextBlob Fallback | Polarity: -0.60)

[2] Title: Nvidia Stock Is Stuck. Taiwan Tech Conference Could Be a Catalyst.
    Date Published: 2026-05-29 12:49:00+00:00
    AI Sentiment Tag: 🟡 NEUTRAL (Local TextBlob Fallback | Polarity: 0.00)

[3] Title: Dell Raises Outlook as AI Server Demand Surges
    Date Published: 2026-05-29 12:22:30+00:00
    AI Sentiment Tag: 🟡 NEUTRAL (Local TextBlob Fallback | Polarity: 0.00)

[4] Title: Is Nvidia a Top Dividend Stock?
    Date Published: 2026-05-29 10:50:00+00:00
    AI Sentiment Tag: 🟢 BULLISH (Local TextBlob Fallback | Polarity: 0.50)

[5] Title: Jim Cramer on NVIDIA: “The King Has Indeed Begun to Underperform When It Reports”
    Date Published: 2026-05-29 10:49: